# 内核重启恢复入口
先只运行下面两格：第 2 格恢复历史特征，第 3 格生成近期特征并合并。跳过原始 CSV 统计和抽样。
历史代码已保存于后方；原始最早创建位置是第 49 格，依赖第 46、48 格。
恢复缓存依据样本路径、大小、修改时间及构建代码校验，失效时仅重新读取样本 Parquet。
近期会话与历史会话统一使用 rebuilt_session（按用户、相邻事件间隔超过30分钟重建），与已保存的15特征模型缓存一致。
第3格会校验近期会话口径，并在模型特征缓存存在时核对全部15个模型输入；不一致会报错停止，不会覆盖模型缓存。
修复后请重启内核并依次运行第2、3格；如果编辑器提示磁盘内容变化，请重新加载本文件。
此恢复表不包含后面另行计算的 R/F/M 分数和 customer_segment；需要分群时再运行对应分群格。


In [ ]:
# 恢复入口：内核重启后先运行此格，不需要运行上方原始数据处理。
from pathlib import Path
import hashlib, json
import duckdb
import pandas as pd
import numpy as np

# Works when Jupyter starts in the repository root or notebooks/ directory.
project_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / 'data/sample/session_events.parquet').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Start Jupyter in the project root or notebooks/; provide data/sample/session_events.parquet first.')
sample_path = project_root / 'data/sample/session_events.parquet'
if not sample_path.is_file():
    raise FileNotFoundError(f'找不到已验证的样本缓存：{sample_path}；请检查项目位置，不要重跑原始数据。')
cache_path = project_root / 'data/sample/user_features_recovery_v1.parquet'
meta_path = cache_path.with_suffix('.json')
build_code = 'rfm = duckdb.sql("""\nWITH purchase_data AS (\n    SELECT\n        user_id,\n\n        -- 最近一次购买时间\n        MAX(event_time) AS last_purchase_time,\n\n        -- 购买 session 数\n        COUNT(DISTINCT rebuilt_session) AS purchase_sessions,\n\n        -- 总消费金额\n        SUM(price) AS total_spend\n\n    FROM \'__SESSION_EVENTS_PARQUET__\'\n\n    WHERE event_time < \'2020-03-01\'\n      AND event_type = \'purchase\'\n\n    GROUP BY user_id\n)\n\nSELECT\n    user_id,\n\n    DATE_DIFF(\n        \'second\',\n        last_purchase_time,\n        TIMESTAMP \'2020-03-01 00:00:00\'\n    ) / 86400.0 AS recency_days,\n\n    purchase_sessions,\n\n    total_spend\n\nFROM purchase_data\nORDER BY user_id\n""").df()\n\nrfm.head(10)\n\nbehavior_features = duckdb.sql("""\nSELECT\n    user_id,\n\n    COUNT(DISTINCT rebuilt_session) AS total_sessions,\n\n    SUM(CASE WHEN event_type = \'view\'\n        THEN 1 ELSE 0 END) AS view_count,\n\n    SUM(CASE WHEN event_type = \'cart\'\n        THEN 1 ELSE 0 END) AS cart_count,\n\n    SUM(CASE WHEN event_type = \'purchase\'\n        THEN 1 ELSE 0 END) AS purchase_count\n\nFROM \'__SESSION_EVENTS_PARQUET__\'\n\nWHERE event_time < \'2020-03-01\'\n\nGROUP BY user_id\n""").df()\n\nbehavior_features.head(10)\n\nuser_features = rfm.merge(\n    behavior_features,\n    on="user_id",\n    how="left"\n)\n\nuser_features.head(10)\n\nuser_features["cart_rate"] = (\n    user_features["cart_count"] /\n    user_features["view_count"]\n)\n\nuser_features["purchase_rate"] = (\n    user_features["purchase_count"] /\n    user_features["view_count"]\n)\n\nuser_features["purchase_session_rate"] = (\n    user_features["purchase_sessions"] /\n    user_features["total_sessions"]\n)\n\nuser_features.head(10)\n\nimport numpy as np\n\n# view_count > 0 时正常计算；\n# 没有 view 记录时设为缺失值 NaN\nuser_features["cart_rate"] = np.where(\n    user_features["view_count"] > 0,\n    user_features["cart_count"] / user_features["view_count"],\n    np.nan\n)\n\nuser_features["purchase_rate"] = np.where(\n    user_features["view_count"] > 0,\n    user_features["purchase_count"] / user_features["view_count"],\n    np.nan\n)\n\n# 标记该用户是否完全没有 view 记录\nuser_features["no_view_before"] = (\n    user_features["view_count"] == 0\n).astype(int)\n\nuser_features.head(10)\n\nuser_features["avg_spend_per_purchase"] = (\n    user_features["total_spend"] /\n    user_features["purchase_count"]\n)\n\nuser_features[\n    [\n        "user_id",\n        "total_spend",\n        "purchase_count",\n        "avg_spend_per_purchase"\n    ]\n].head(10)\n\nactivity_features = duckdb.sql("""\nWITH ranked_events AS (\n    SELECT\n        user_id,\n        event_time,\n        event_type,\n\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY event_time DESC\n        ) AS rn\n\n    FROM \'__SESSION_EVENTS_PARQUET__\'\n\n    WHERE event_time < \'2020-03-01\'\n)\n\nSELECT\n    user_id,\n\n    DATE_DIFF(\n        \'second\',\n        event_time,\n        TIMESTAMP \'2020-03-01 00:00:00\'\n    ) / 86400.0 AS last_activity_days,\n\n    event_type AS last_event_type\n\nFROM ranked_events\n\nWHERE rn = 1\n""").df()\n\nactivity_features.head(10)\n\nuser_features = user_features.merge(\n    activity_features,\n    on="user_id",\n    how="left"\n)\n\nuser_features[\n    [\n        "user_id",\n        "recency_days",\n        "last_activity_days",\n        "last_event_type"\n    ]\n].head(10)\n\nuser_labels = duckdb.sql("""\nWITH eligible AS (\n    SELECT DISTINCT user_id\n    FROM \'__SESSION_EVENTS_PARQUET__\'\n    WHERE event_time < \'2020-03-01\'\n      AND event_type = \'purchase\'\n),\n\nfuture_behavior AS (\n    SELECT\n        user_id,\n        COUNT(*) AS future_events,\n        SUM(\n            CASE WHEN event_type = \'purchase\'\n            THEN 1 ELSE 0 END\n        ) AS future_purchases\n\n    FROM \'__SESSION_EVENTS_PARQUET__\'\n\n    WHERE event_time >= \'2020-03-01\'\n\n    GROUP BY user_id\n)\n\nSELECT\n    e.user_id,\n\n    CASE\n        WHEN COALESCE(f.future_purchases, 0) > 0\n            THEN \'retained\'\n        WHEN COALESCE(f.future_events, 0) > 0\n            THEN \'active_non_buyer\'\n        ELSE \'inactive\'\n    END AS future_status\n\nFROM eligible e\n\nLEFT JOIN future_behavior f\n    ON e.user_id = f.user_id\n""").df()\n\nuser_labels["churn"] = (\n    user_labels["future_status"] != "retained"\n).astype(int)\n\nuser_features = user_features.merge(\n    user_labels,\n    on="user_id",\n    how="left"\n)\n\nuser_features[\n    ["user_id", "future_status", "churn"]\n].head(10)'.replace('__SESSION_EVENTS_PARQUET__', str(sample_path))
signature = {'source': str(sample_path), 'size': sample_path.stat().st_size,
             'mtime_ns': sample_path.stat().st_mtime_ns,
             'code_sha256': hashlib.sha256(build_code.encode()).hexdigest()}
if cache_path.exists() and meta_path.exists() and json.loads(meta_path.read_text()) == signature:
    user_features = duckdb.sql('SELECT * FROM read_parquet(?)', params=[str(cache_path)]).df()
    print('已读取用户特征缓存')
else:
    exec(compile(build_code, '<original-feature-cells>', 'exec'))
    assert user_features.user_id.is_unique and user_features.user_id.notna().all()
    assert user_features.churn.notna().all()
    assert not any(c.endswith(('_x', '_y')) for c in user_features.columns)
    temp_path = cache_path.with_suffix('.tmp.parquet')
    duckdb.sql('SELECT * FROM user_features').write_parquet(str(temp_path))
    temp_path.replace(cache_path)
    meta_path.write_text(json.dumps(signature, indent=2))
    print('已从 session_events.parquet 按需重建并缓存')
user_labels = user_features[['user_id', 'future_status', 'churn']].copy()
assert user_features.user_id.is_unique
print('user_features:', user_features.shape)
print(user_features.churn.value_counts().sort_index())
display(user_features.head())


In [ ]:
# 恢复近期特征：与历史及15特征模型统一使用重建会话。
# 重建规则：按 user_id 分组，首条事件或相邻事件间隔 > 30 分钟开始新会话。
# 不使用原始 user_session；该字段与模型口径不同。
recent_session_column = "rebuilt_session"
assert recent_session_column == "rebuilt_session", "近期会话必须使用 rebuilt_session，与历史和已保存模型一致。"
recent_activity = duckdb.sql(f"""
SELECT user_id,
 SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS recent_30d_views,
 SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS recent_30d_carts,
 SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS recent_30d_purchases,
 COUNT(DISTINCT {recent_session_column}) AS recent_30d_sessions
FROM read_parquet(?)
WHERE event_time >= '2020-01-31' AND event_time < '2020-03-01'
GROUP BY user_id
""", params=[str(sample_path)]).df()
recent_cols = [c for c in recent_activity.columns if c != 'user_id']
# 允许重复运行本格，也避免旧近期列形成 _x / _y 后缀。
recovered_features = (
    user_features.drop(columns=recent_cols, errors='ignore')
    .merge(recent_activity, on='user_id', how='left', validate='one_to_one')
)
recovered_features[recent_cols] = recovered_features[recent_cols].fillna(0)
assert len(recovered_features) == len(user_features)
assert recovered_features.user_id.is_unique
assert not any(c.endswith(('_x', '_y')) for c in recovered_features.columns)
assert (recovered_features.recent_30d_sessions <= recovered_features.total_sessions).all(), (
    "近期会话口径不一致：近期会话数不能超过历史重建会话数。请使用 rebuilt_session。"
)

# 从事件缓存独立枚举近期的 (user_id, rebuilt_session) 对，防止会话字段误改。
recent_pairs = duckdb.sql("""
SELECT DISTINCT user_id, rebuilt_session
FROM read_parquet(?)
WHERE event_time >= '2020-01-31' AND event_time < '2020-03-01'
""", params=[str(sample_path)]).df()
assert recent_pairs[['user_id', 'rebuilt_session']].notna().all().all()
expected_recent_sessions = (
    recent_pairs.groupby('user_id').size()
    .reindex(recovered_features.user_id, fill_value=0).to_numpy()
)
assert np.array_equal(recovered_features.recent_30d_sessions.to_numpy(), expected_recent_sessions), (
    "近期会话口径不一致：必须按 user_id 统计 DISTINCT rebuilt_session。请勿使用 user_session。"
)

# 只读核对已保存的模型输入；绝不覆盖已验证的特征缓存或模型。
model_feature_cols = [
    'recency_days', 'purchase_sessions', 'total_spend', 'total_sessions',
    'view_count', 'cart_count', 'purchase_count', 'cart_rate', 'purchase_rate',
    'no_view_before', 'purchase_session_rate', 'recent_30d_views',
    'recent_30d_carts', 'recent_30d_purchases', 'recent_30d_sessions'
]
model_reference_path = project_root / 'data/sample/user_features_v2.parquet'
if model_reference_path.is_file():
    reference_features = duckdb.sql('SELECT * FROM read_parquet(?)', params=[str(model_reference_path)]).df()
    assert reference_features.user_id.is_unique
    assert set(reference_features.user_id) == set(recovered_features.user_id), (
        "恢复数据与已保存模型缓存的用户集合不同；请先核查数据版本，不要覆盖缓存。"
    )
    actual = recovered_features.set_index('user_id')[model_feature_cols].copy()
    expected = reference_features.set_index('user_id').loc[actual.index, model_feature_cols].copy()
    ratio_cols = ['cart_rate', 'purchase_rate']
    actual[ratio_cols] = actual[ratio_cols].fillna(0)
    expected[ratio_cols] = expected[ratio_cols].fillna(0)
    mismatched_cols = [c for c in model_feature_cols if not np.allclose(
        actual[c].to_numpy(), expected[c].to_numpy(), rtol=1e-10, atol=1e-8, equal_nan=False
    )]
    assert not mismatched_cols, (
        f"恢复特征与已保存模型缓存不一致：{mismatched_cols}。请核查数据版本或特征口径，不要覆盖缓存。"
    )
    print('15个模型输入与已保存特征缓存一致（按 user_id 核对）。')
else:
    print('未找到已保存模型特征缓存；已通过 rebuilt_session 事件级校验，未进行15特征参考比对。')

# 所有校验通过后才替换下游使用的恢复表。
user_features_v2 = recovered_features
print('近期会话口径：rebuilt_session；user_features_v2:', user_features_v2.shape)
display(user_features_v2[['user_id'] + recent_cols].head(10))


# REES46 E-commerce Data Quality Audit

## 1. Load and Inspect Raw Data

This notebook examines the structure and quality of the raw REES46 e-commerce event data before further analysis.

In [ ]:
import pandas as pd

In [ ]:
file_path = "../data/raw/2019-Oct.csv.gz"

df_preview = pd.read_csv(
    file_path,
    compression="gzip",
    nrows=10
)

df_preview

In [ ]:
df_preview.columns.tolist()

In [ ]:
df_preview.info()

## 2. Create a Development Sample

A manageable sample is created from the raw event data for exploratory analysis and development.

In [ ]:
import duckdb

print("DuckDB version:", duckdb.__version__)

In [ ]:
query = """
SELECT *
FROM read_csv_auto('../data/raw/2019-Oct.csv.gz')
LIMIT 5
"""

duckdb.sql(query).df()

In [ ]:
query = """
SELECT COUNT(DISTINCT user_id) AS total_users
FROM read_csv_auto('../data/raw/2019-Oct.csv.gz')
"""

duckdb.sql(query).df()

In [ ]:
query = """
SELECT COUNT(*) AS total_events
FROM read_csv_auto('../data/raw/2019-Oct.csv.gz')
"""

duckdb.sql(query).df()

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

raw_dir = Path("../data/raw")

months = [
    ("2019-10", "2019-Oct.csv.gz"),
    ("2019-11", "2019-Nov.csv.gz"),
    ("2019-12", "2019-Dec.csv.gz"),
    ("2020-01", "2020-Jan.csv.gz"),
    ("2020-02", "2020-Feb.csv.gz"),
    ("2020-03", "2020-Mar.csv.gz"),
    ("2020-04", "2020-Apr.csv.gz"),
]

results = []

with duckdb.connect() as con:
    for month, filename in months:
        print(f"正在统计：{month}", flush=True)

        event_count = con.execute(
            """
            SELECT COUNT(*)
            FROM read_csv_auto(?, header = true)
            """,
            [str(raw_dir / filename)]
        ).fetchone()[0]

        results.append({
            "month": month,
            "event_count": event_count
        })

monthly_counts = pd.DataFrame(results)
monthly_counts

In [ ]:
query = """
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT user_id) AS total_users,
    MIN(event_time) AS start_time,
    MAX(event_time) AS end_time,

    COUNT(*) FILTER (WHERE event_type = 'view') AS view_events,
    COUNT(*) FILTER (WHERE event_type = 'cart') AS cart_events,
    COUNT(*) FILTER (WHERE event_type = 'remove_from_cart') AS remove_from_cart_events,
    COUNT(*) FILTER (WHERE event_type = 'purchase') AS purchase_events,

    COUNT(*) FILTER (WHERE user_id IS NULL) AS missing_user_id,
    COUNT(*) FILTER (WHERE event_time IS NULL) AS missing_event_time,
    COUNT(*) FILTER (WHERE event_type IS NULL) AS missing_event_type

FROM read_csv_auto('../data/raw/*.csv.gz')
"""

overview = duckdb.sql(query).df()
overview

In [ ]:
query = """
WITH user_event_counts AS (
    SELECT
        user_id,
        COUNT(*) AS event_count
    FROM read_csv_auto('../data/raw/*.csv.gz')
    GROUP BY user_id
)

SELECT
    COUNT(*) AS total_users,
    AVG(event_count) AS avg_events,
    MEDIAN(event_count) AS median_events,
    QUANTILE_CONT(event_count, 0.25) AS p25_events,
    QUANTILE_CONT(event_count, 0.75) AS p75_events,
    QUANTILE_CONT(event_count, 0.90) AS p90_events,
    QUANTILE_CONT(event_count, 0.95) AS p95_events,
    MAX(event_count) AS max_events
FROM user_event_counts
"""

user_distribution = duckdb.sql(query).df()
user_distribution

In [ ]:
sample_query = """
WITH all_users AS (
    SELECT DISTINCT user_id
    FROM read_csv_auto('../data/raw/*.csv.gz')
),

sampled_users AS (
    SELECT user_id
    FROM all_users
    ORDER BY hash(user_id)
    LIMIT 100000
)

SELECT events.*
FROM read_csv_auto('../data/raw/*.csv.gz') AS events
INNER JOIN sampled_users
    ON events.user_id = sampled_users.user_id
"""

sample_df = duckdb.sql(sample_query).df()

sample_df.shape

In [ ]:
duckdb.sql("""
COPY sample_df
TO '../data/sample/development_sample.parquet'
(FORMAT PARQUET)
""")

In [ ]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT user_id) AS total_users
FROM '../data/sample/development_sample.parquet'
""").df()

## 3. User Behavior Analysis

This section analyzes how users interact with the e-commerce platform through viewing, adding products to cart, and purchasing.

In [ ]:
query = """
SELECT
    event_type,
    COUNT(*) AS event_count
FROM '../data/sample/development_sample.parquet'
GROUP BY event_type
ORDER BY event_count DESC
"""

event_counts = duckdb.sql(query).df()
event_counts

In [ ]:
event_counts["percentage"] = (
    event_counts["event_count"]
    / event_counts["event_count"].sum()
    * 100
)

event_counts

In [ ]:
query = """
SELECT
    event_type,
    COUNT(DISTINCT user_id) AS user_count
FROM '../data/sample/development_sample.parquet'
GROUP BY event_type
ORDER BY user_count DESC
"""

funnel_users = duckdb.sql(query).df()
funnel_users

In [ ]:
query = """
WITH user_behavior AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS has_view,
        MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM '../data/sample/development_sample.parquet'
    GROUP BY user_id
)

SELECT
    has_view,
    has_cart,
    has_purchase,
    COUNT(*) AS user_count
FROM user_behavior
GROUP BY has_view, has_cart, has_purchase
ORDER BY user_count DESC
"""

behavior_combinations = duckdb.sql(query).df()
behavior_combinations

## 4. Session-Level User Journey Analysis

This section analyzes user behavior within individual sessions to understand the sequence from product viewing to cart addition and purchase.

In [ ]:
query = """
WITH purchase_sessions AS (
    SELECT DISTINCT user_session
    FROM '../data/sample/development_sample.parquet'
    WHERE event_type = 'purchase'
      AND user_session IS NOT NULL
    LIMIT 3
)

SELECT
    event_time,
    user_id,
    user_session,
    product_id,
    event_type,
    price
FROM '../data/sample/development_sample.parquet'
WHERE user_session IN (
    SELECT user_session
    FROM purchase_sessions
)
ORDER BY user_session, event_time
"""

session_examples = duckdb.sql(query).df()
session_examples

In [ ]:
query = """
SELECT
    user_id,
    user_session,
    product_id,

    MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS viewed,
    MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carted,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchased,

    MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
    MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
    MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time

FROM '../data/sample/development_sample.parquet'

WHERE user_session IS NOT NULL

GROUP BY
    user_id,
    user_session,
    product_id
"""

journey = duckdb.sql(query).df()
journey.head(10)

In [ ]:
journey[journey["purchased"] == 1].head(10)

In [ ]:
def classify_path(row):
    if row["viewed"] == 1 and row["carted"] == 0 and row["purchased"] == 0:
        return "view_only"

    elif (
        row["viewed"] == 1
        and row["carted"] == 1
        and row["purchased"] == 0
        and row["first_view_time"] < row["first_cart_time"]
    ):
        return "view_to_cart_no_purchase"

    elif (
        row["viewed"] == 1
        and row["carted"] == 0
        and row["purchased"] == 1
        and row["first_view_time"] < row["first_purchase_time"]
    ):
        return "view_to_purchase"

    elif (
        row["viewed"] == 1
        and row["carted"] == 1
        and row["purchased"] == 1
        and row["first_view_time"] < row["first_cart_time"] < row["first_purchase_time"]
    ):
        return "view_to_cart_to_purchase"

    else:
        return "other"


journey["path"] = journey.apply(classify_path, axis=1)

journey["path"].value_counts()

In [ ]:
journey[journey["path"] == "other"][
    [
        "viewed",
        "carted",
        "purchased",
        "first_view_time",
        "first_cart_time",
        "first_purchase_time"
    ]
].head(20)

In [ ]:
query = """
WITH ordered_events AS (
    SELECT
        user_id,
        event_time,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time
    FROM '../data/sample/development_sample.parquet'
),

gaps AS (
    SELECT
        DATE_DIFF(
            'minute',
            previous_event_time,
            event_time
        ) AS gap_minutes
    FROM ordered_events
    WHERE previous_event_time IS NOT NULL
)

SELECT
    MEDIAN(gap_minutes) AS median_gap,
    QUANTILE_CONT(gap_minutes, 0.75) AS p75_gap,
    QUANTILE_CONT(gap_minutes, 0.90) AS p90_gap,
    QUANTILE_CONT(gap_minutes, 0.95) AS p95_gap,
    QUANTILE_CONT(gap_minutes, 0.99) AS p99_gap
FROM gaps
"""

gap_distribution = duckdb.sql(query).df()
gap_distribution

In [ ]:
query = """
WITH ordered_events AS (
    SELECT
        user_id,
        event_time,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time
    FROM '../data/sample/development_sample.parquet'
),

gaps AS (
    SELECT
        DATE_DIFF(
            'second',
            previous_event_time,
            event_time
        ) / 60.0 AS gap_minutes
    FROM ordered_events
    WHERE previous_event_time IS NOT NULL
)

SELECT
    COUNT(*) AS total_gaps,

    ROUND(100.0 * AVG(CASE WHEN gap_minutes <= 5 THEN 1 ELSE 0 END), 2) AS within_5min,
    ROUND(100.0 * AVG(CASE WHEN gap_minutes <= 10 THEN 1 ELSE 0 END), 2) AS within_10min,
    ROUND(100.0 * AVG(CASE WHEN gap_minutes <= 20 THEN 1 ELSE 0 END), 2) AS within_20min,
    ROUND(100.0 * AVG(CASE WHEN gap_minutes <= 30 THEN 1 ELSE 0 END), 2) AS within_30min,
    ROUND(100.0 * AVG(CASE WHEN gap_minutes <= 60 THEN 1 ELSE 0 END), 2) AS within_60min,
    ROUND(100.0 * AVG(CASE WHEN gap_minutes <= 120 THEN 1 ELSE 0 END), 2) AS within_120min

FROM gaps
"""

gap_thresholds = duckdb.sql(query).df()
gap_thresholds

In [ ]:
query = """
WITH ordered_events AS (
    SELECT
        user_id,
        event_time,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time
    FROM '../data/sample/development_sample.parquet'
),

gaps AS (
    SELECT
        DATE_DIFF(
            'second',
            previous_event_time,
            event_time
        ) / 60.0 AS gap_minutes
    FROM ordered_events
    WHERE previous_event_time IS NOT NULL
),

gap_buckets AS (
    SELECT
        CASE
            WHEN gap_minutes <= 5 THEN '01. 0-5 min'
            WHEN gap_minutes <= 10 THEN '02. 5-10 min'
            WHEN gap_minutes <= 20 THEN '03. 10-20 min'
            WHEN gap_minutes <= 30 THEN '04. 20-30 min'
            WHEN gap_minutes <= 60 THEN '05. 30-60 min'
            WHEN gap_minutes <= 120 THEN '06. 1-2 hours'
            WHEN gap_minutes <= 360 THEN '07. 2-6 hours'
            WHEN gap_minutes <= 1440 THEN '08. 6-24 hours'
            WHEN gap_minutes <= 10080 THEN '09. 1-7 days'
            ELSE '10. >7 days'
        END AS gap_bucket
    FROM gaps
)

SELECT
    gap_bucket,
    COUNT(*) AS gap_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM gap_buckets
GROUP BY gap_bucket
ORDER BY gap_bucket
"""

gap_buckets = duckdb.sql(query).df()
gap_buckets

In [ ]:
query = """
WITH ordered_events AS (
    SELECT
        *,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time
    FROM '../data/sample/development_sample.parquet'
)

SELECT
    *,
    CASE
        WHEN previous_event_time IS NULL THEN 1
        WHEN DATE_DIFF(
            'second',
            previous_event_time,
            event_time
        ) > 30 * 60 THEN 1
        ELSE 0
    END AS new_session
FROM ordered_events
ORDER BY user_id, event_time
"""

session_events = duckdb.sql(query).df()

session_events[
    [
        "user_id",
        "event_time",
        "event_type",
        "product_id",
        "previous_event_time",
        "new_session"
    ]
].head(20)

In [ ]:
session_events["rebuilt_session"] = (
    session_events
    .groupby("user_id")["new_session"]
    .cumsum()
)

session_events[
    [
        "user_id",
        "event_time",
        "event_type",
        "product_id",
        "new_session",
        "rebuilt_session"
    ]
].head(20)

In [ ]:
duckdb.sql("""
COPY session_events
TO '../data/sample/session_events.parquet'
(FORMAT PARQUET)
""")

In [ ]:
check = duckdb.sql("""
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(*) FILTER (WHERE new_session = 1) AS total_sessions
FROM '../data/sample/session_events.parquet'
""").df()

check

In [ ]:
query = """
SELECT
    user_id,
    rebuilt_session,
    product_id,

    MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS viewed,
    MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carted,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchased,

    MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
    MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
    MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time

FROM '../data/sample/session_events.parquet'

GROUP BY
    user_id,
    rebuilt_session,
    product_id
"""

rebuilt_journey = duckdb.sql(query).df()

rebuilt_journey.head(10)

In [ ]:
def classify_path(row):
    if row["viewed"] == 1 and row["carted"] == 0 and row["purchased"] == 0:
        return "view_only"

    elif (
        row["viewed"] == 1
        and row["carted"] == 1
        and row["purchased"] == 0
        and row["first_view_time"] < row["first_cart_time"]
    ):
        return "view_to_cart_no_purchase"

    elif (
        row["viewed"] == 1
        and row["carted"] == 0
        and row["purchased"] == 1
        and row["first_view_time"] < row["first_purchase_time"]
    ):
        return "view_to_purchase"

    elif (
        row["viewed"] == 1
        and row["carted"] == 1
        and row["purchased"] == 1
        and row["first_view_time"] < row["first_cart_time"] < row["first_purchase_time"]
    ):
        return "view_to_cart_to_purchase"

    else:
        return "other"


rebuilt_journey["path"] = rebuilt_journey.apply(classify_path, axis=1)

rebuilt_journey["path"].value_counts()

In [ ]:
query = """
WITH product_paths AS (
    SELECT
        user_id,
        rebuilt_session,
        product_id,

        STRING_AGG(
            event_type,
            ' -> '
            ORDER BY event_time
        ) AS behavior_sequence

    FROM '../data/sample/session_events.parquet'

    GROUP BY
        user_id,
        rebuilt_session,
        product_id
)

SELECT
    behavior_sequence,
    COUNT(*) AS path_count
FROM product_paths

GROUP BY behavior_sequence
ORDER BY path_count DESC
"""

actual_paths = duckdb.sql(query).df()

actual_paths.head(30)

In [ ]:
query = """
WITH purchase_sessions AS (
    SELECT
        user_id,
        rebuilt_session,
        MIN(event_time) AS purchase_time
    FROM '../data/sample/session_events.parquet'
    WHERE event_type = 'purchase'
    GROUP BY user_id, rebuilt_session
),

purchase_history AS (
    SELECT
        user_id,
        rebuilt_session,
        purchase_time,

        LAG(purchase_time) OVER (
            PARTITION BY user_id
            ORDER BY purchase_time
        ) AS previous_purchase_time

    FROM purchase_sessions
)

SELECT
    user_id,
    rebuilt_session,
    purchase_time,
    previous_purchase_time,

    DATE_DIFF(
        'second',
        previous_purchase_time,
        purchase_time
    ) / 86400.0 AS purchase_gap_days

FROM purchase_history

WHERE previous_purchase_time IS NOT NULL
"""

purchase_gaps = duckdb.sql(query).df()

purchase_gaps.head(10)

In [ ]:

purchase_gap_summary = duckdb.sql("""
SELECT
    COUNT(*) AS total_repeat_purchases,

    ROUND(AVG(purchase_gap_days), 2) AS mean_days,
    ROUND(MEDIAN(purchase_gap_days), 2) AS median_days,

    ROUND(QUANTILE_CONT(purchase_gap_days, 0.25), 2) AS p25,
    ROUND(QUANTILE_CONT(purchase_gap_days, 0.75), 2) AS p75,
    ROUND(QUANTILE_CONT(purchase_gap_days, 0.90), 2) AS p90,
    ROUND(QUANTILE_CONT(purchase_gap_days, 0.95), 2) AS p95,

    ROUND(
        100.0 * AVG(CASE WHEN purchase_gap_days <= 1 THEN 1 ELSE 0 END),
        2
    ) AS within_1_day,

    ROUND(
        100.0 * AVG(CASE WHEN purchase_gap_days <= 7 THEN 1 ELSE 0 END),
        2
    ) AS within_7_days,

    ROUND(
        100.0 * AVG(CASE WHEN purchase_gap_days <= 30 THEN 1 ELSE 0 END),
        2
    ) AS within_30_days,

    ROUND(
        100.0 * AVG(CASE WHEN purchase_gap_days <= 60 THEN 1 ELSE 0 END),
        2
    ) AS within_60_days,

    ROUND(
        100.0 * AVG(CASE WHEN purchase_gap_days <= 90 THEN 1 ELSE 0 END),
        2
    ) AS within_90_days

FROM purchase_gaps
""").df()

purchase_gap_summary

In [ ]:
date_range = duckdb.sql("""
SELECT
    MIN(event_time) AS data_start,
    MAX(event_time) AS data_end
FROM '../data/sample/session_events.parquet'
""").df()

date_range

In [ ]:
eligible_users = duckdb.sql("""
SELECT
    COUNT(DISTINCT user_id) AS eligible_users
FROM '../data/sample/session_events.parquet'
WHERE event_time < '2020-03-01'
  AND event_type = 'purchase'
""").df()

eligible_users

In [ ]:
future_status = duckdb.sql("""
WITH eligible AS (
    SELECT DISTINCT user_id
    FROM '../data/sample/session_events.parquet'
    WHERE event_time < '2020-03-01'
      AND event_type = 'purchase'
),

future_behavior AS (
    SELECT
        user_id,
        COUNT(*) AS future_events,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)
            AS future_purchases
    FROM '../data/sample/session_events.parquet'
    WHERE event_time >= '2020-03-01'
    GROUP BY user_id
)

SELECT
    CASE
        WHEN COALESCE(f.future_purchases, 0) > 0
            THEN 'retained'

        WHEN COALESCE(f.future_events, 0) > 0
            THEN 'active_non_buyer'

        ELSE 'inactive'
    END AS future_status,

    COUNT(*) AS user_count

FROM eligible e
LEFT JOIN future_behavior f
    ON e.user_id = f.user_id

GROUP BY future_status
ORDER BY user_count DESC
""").df()

future_status

# 5. Churn Definition and User-Level Feature Engineering

This section defines the churn target using historical purchase behavior and constructs user-level features for subsequent churn analysis and modeling.

In [ ]:
rfm = duckdb.sql("""
WITH purchase_data AS (
    SELECT
        user_id,

        -- 最近一次购买时间
        MAX(event_time) AS last_purchase_time,

        -- 购买 session 数
        COUNT(DISTINCT rebuilt_session) AS purchase_sessions,

        -- 总消费金额
        SUM(price) AS total_spend

    FROM '../data/sample/session_events.parquet'

    WHERE event_time < '2020-03-01'
      AND event_type = 'purchase'

    GROUP BY user_id
)

SELECT
    user_id,

    DATE_DIFF(
        'second',
        last_purchase_time,
        TIMESTAMP '2020-03-01 00:00:00'
    ) / 86400.0 AS recency_days,

    purchase_sessions,

    total_spend

FROM purchase_data
ORDER BY user_id
""").df()

rfm.head(10)

In [ ]:
rfm.shape

In [ ]:
behavior_features = duckdb.sql("""
SELECT
    user_id,

    COUNT(DISTINCT rebuilt_session) AS total_sessions,

    SUM(CASE WHEN event_type = 'view'
        THEN 1 ELSE 0 END) AS view_count,

    SUM(CASE WHEN event_type = 'cart'
        THEN 1 ELSE 0 END) AS cart_count,

    SUM(CASE WHEN event_type = 'purchase'
        THEN 1 ELSE 0 END) AS purchase_count

FROM '../data/sample/session_events.parquet'

WHERE event_time < '2020-03-01'

GROUP BY user_id
""").df()

behavior_features.head(10)

In [ ]:
user_features = rfm.merge(
    behavior_features,
    on="user_id",
    how="left"
)

user_features.head(10)

In [ ]:
user_features.shape

In [ ]:
user_features["cart_rate"] = (
    user_features["cart_count"] /
    user_features["view_count"]
)

user_features["purchase_rate"] = (
    user_features["purchase_count"] /
    user_features["view_count"]
)

user_features["purchase_session_rate"] = (
    user_features["purchase_sessions"] /
    user_features["total_sessions"]
)

user_features.head(10)

In [ ]:
user_features[
    (user_features["view_count"] == 0) |
    user_features["cart_rate"].isna() |
    user_features["purchase_rate"].isna()
]

In [ ]:
import numpy as np

# view_count > 0 时正常计算；
# 没有 view 记录时设为缺失值 NaN
user_features["cart_rate"] = np.where(
    user_features["view_count"] > 0,
    user_features["cart_count"] / user_features["view_count"],
    np.nan
)

user_features["purchase_rate"] = np.where(
    user_features["view_count"] > 0,
    user_features["purchase_count"] / user_features["view_count"],
    np.nan
)

# 标记该用户是否完全没有 view 记录
user_features["no_view_before"] = (
    user_features["view_count"] == 0
).astype(int)

user_features.head(10)

In [ ]:
import numpy as np

print(
    "cart_rate inf:",
    np.isinf(user_features["cart_rate"]).sum()
)

print(
    "purchase_rate inf:",
    np.isinf(user_features["purchase_rate"]).sum()
)

print(
    "no_view users:",
    user_features["no_view_before"].sum()
)

In [ ]:
user_features["avg_spend_per_purchase"] = (
    user_features["total_spend"] /
    user_features["purchase_count"]
)

user_features[
    [
        "user_id",
        "total_spend",
        "purchase_count",
        "avg_spend_per_purchase"
    ]
].head(10)

In [ ]:
activity_features = duckdb.sql("""
WITH ranked_events AS (
    SELECT
        user_id,
        event_time,
        event_type,

        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_time DESC
        ) AS rn

    FROM '../data/sample/session_events.parquet'

    WHERE event_time < '2020-03-01'
)

SELECT
    user_id,

    DATE_DIFF(
        'second',
        event_time,
        TIMESTAMP '2020-03-01 00:00:00'
    ) / 86400.0 AS last_activity_days,

    event_type AS last_event_type

FROM ranked_events

WHERE rn = 1
""").df()

activity_features.head(10)

In [ ]:
user_features = user_features.drop(columns=[c for c in activity_features.columns if c != "user_id"], errors="ignore").merge(
    activity_features,
    on="user_id",
    how="left",
    validate="one_to_one"
)

user_features[
    [
        "user_id",
        "recency_days",
        "last_activity_days",
        "last_event_type"
    ]
].head(10)

In [ ]:
%whos DataFrame

In [ ]:
user_labels = duckdb.sql("""
WITH eligible AS (
    SELECT DISTINCT user_id
    FROM '../data/sample/session_events.parquet'
    WHERE event_time < '2020-03-01'
      AND event_type = 'purchase'
),

future_behavior AS (
    SELECT
        user_id,
        COUNT(*) AS future_events,
        SUM(
            CASE WHEN event_type = 'purchase'
            THEN 1 ELSE 0 END
        ) AS future_purchases

    FROM '../data/sample/session_events.parquet'

    WHERE event_time >= '2020-03-01'

    GROUP BY user_id
)

SELECT
    e.user_id,

    CASE
        WHEN COALESCE(f.future_purchases, 0) > 0
            THEN 'retained'
        WHEN COALESCE(f.future_events, 0) > 0
            THEN 'active_non_buyer'
        ELSE 'inactive'
    END AS future_status

FROM eligible e

LEFT JOIN future_behavior f
    ON e.user_id = f.user_id
""").df()

user_labels["churn"] = (
    user_labels["future_status"] != "retained"
).astype(int)

user_features = user_features.drop(columns=[c for c in user_labels.columns if c != "user_id"], errors="ignore").merge(
    user_labels,
    on="user_id",
    how="left",
    validate="one_to_one"
)

user_features[
    ["user_id", "future_status", "churn"]
].head(10)

In [ ]:
user_labels.head(10)

In [ ]:
print(user_features.shape)

print(
    user_features["future_status"].value_counts()
)

print(
    user_features["churn"].value_counts()
)

# 6. Churn EDA and User Profiling

Compare historical behavioral characteristics between retained and churned users to identify potential churn signals.

In [ ]:
eda_compare = user_features.groupby("churn")[
    [
        "recency_days",
        "purchase_session_rate",
        "cart_rate"
    ]
].median()

eda_compare

In [ ]:
user_features.groupby("churn")[
    ["view_count", "cart_count"]
].median()

In [ ]:
monthly_churn_trend = duckdb.sql("""
SELECT
    DATE_TRUNC('month', s.event_time) AS month,

    COUNT(DISTINCT s.user_session) AS sessions,
    
    SUM(CASE WHEN s.event_type = 'view' THEN 1 ELSE 0 END) AS views,
    
    SUM(CASE WHEN s.event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
    
    SUM(CASE WHEN s.event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases

FROM '../data/sample/session_events.parquet' s

INNER JOIN user_labels u
    ON s.user_id = u.user_id

WHERE u.churn = 1
  AND s.event_time < '2020-03-01'

GROUP BY month
ORDER BY month
""").df()

monthly_churn_trend

In [ ]:
user_monthly_behavior = duckdb.sql("""
SELECT
    s.user_id,
    DATE_TRUNC('month', s.event_time) AS month,

    SUM(
        CASE WHEN s.event_type = 'view'
        THEN 1 ELSE 0 END
    ) AS views,

    SUM(
        CASE WHEN s.event_type = 'cart'
        THEN 1 ELSE 0 END
    ) AS carts,

    SUM(
        CASE WHEN s.event_type = 'purchase'
        THEN 1 ELSE 0 END
    ) AS purchases

FROM '../data/sample/session_events.parquet' s

INNER JOIN user_labels u
    ON s.user_id = u.user_id

WHERE u.churn = 1
  AND s.event_time < '2020-03-01'

GROUP BY
    s.user_id,
    month

ORDER BY
    s.user_id,
    month
""").df()

user_monthly_behavior.head(15)

In [ ]:
import pandas as pd
import numpy as np

# 5个特征期月份
months = pd.date_range(
    "2019-10-01",
    "2020-02-01",
    freq="MS"
)

# 7658名流失用户
churn_users = user_labels.loc[
    user_labels["churn"] == 1,
    "user_id"
]

# 建立“每个用户 × 每个月”的完整框架
full_index = pd.MultiIndex.from_product(
    [churn_users, months],
    names=["user_id", "month"]
)

monthly_complete = (
    user_monthly_behavior
    .set_index(["user_id", "month"])
    .reindex(full_index)
    .reset_index()
)

# 找到每个用户第一次真正出现行为的月份
first_seen = (
    user_monthly_behavior
    .groupby("user_id")["month"]
    .min()
)

monthly_complete["first_seen"] = (
    monthly_complete["user_id"].map(first_seen)
)

# 首次出现之后的空月份补0；
# 首次出现之前仍然保持NaN
behavior_cols = ["views", "carts", "purchases"]

mask = monthly_complete["month"] >= monthly_complete["first_seen"]

monthly_complete.loc[mask, behavior_cols] = (
    monthly_complete.loc[mask, behavior_cols].fillna(0)
)

monthly_complete.head(15)

In [ ]:
import numpy as np
import pandas as pd

def calculate_slope(group, column):
    valid = group.dropna(subset=[column])

    if len(valid) < 3:
        return np.nan

    x = np.arange(len(valid))
    y = valid[column].values

    return np.polyfit(x, y, 1)[0]


user_slopes = (
    monthly_complete
    .groupby("user_id")
    .apply(
        lambda g: pd.Series({
            "view_slope": calculate_slope(g, "views"),
            "cart_slope": calculate_slope(g, "carts"),
            "purchase_slope": calculate_slope(g, "purchases")
        }),
        include_groups=False
    )
    .reset_index()
)

user_slopes.head(10)

In [ ]:
valid_view_slopes = user_slopes["view_slope"].dropna()

view_trend = pd.Series({
    "decreasing": (valid_view_slopes < -1e-6).sum(),
    "stable": (valid_view_slopes.abs() <= 1e-6).sum(),
    "increasing": (valid_view_slopes > 1e-6).sum()
})

view_trend_percent = (
    view_trend / len(valid_view_slopes) * 100
).round(2)

print("Valid users:", len(valid_view_slopes))
print("\nUser count:")
print(view_trend)

print("\nPercentage:")
print(view_trend_percent)

In [ ]:
retained_monthly = duckdb.sql("""
SELECT
    s.user_id,
    DATE_TRUNC('month', s.event_time) AS month,

    SUM(CASE WHEN s.event_type = 'view'
        THEN 1 ELSE 0 END) AS views

FROM '../data/sample/session_events.parquet' s

INNER JOIN user_labels u
    ON s.user_id = u.user_id

WHERE u.churn = 0
  AND s.event_time < '2020-03-01'

GROUP BY
    s.user_id,
    month

ORDER BY
    s.user_id,
    month
""").df()

retained_monthly.head(10)

In [ ]:
# 建立 retained 用户 × 5个月 的完整框架
retained_users = user_labels.loc[
    user_labels["churn"] == 0,
    "user_id"
]

retained_index = pd.MultiIndex.from_product(
    [retained_users, months],
    names=["user_id", "month"]
)

retained_complete = (
    retained_monthly
    .set_index(["user_id", "month"])
    .reindex(retained_index)
    .reset_index()
)

# 找到每个用户第一次被观察到的月份
retained_first_seen = (
    retained_monthly
    .groupby("user_id")["month"]
    .min()
)

retained_complete["first_seen"] = (
    retained_complete["user_id"].map(retained_first_seen)
)

# 首次出现后的空月份补0
mask = (
    retained_complete["month"]
    >= retained_complete["first_seen"]
)

retained_complete.loc[mask, "views"] = (
    retained_complete.loc[mask, "views"].fillna(0)
)

# 计算每个用户的浏览趋势斜率
retained_view_slopes = (
    retained_complete
    .groupby("user_id")
    .apply(
        lambda g: calculate_slope(g, "views"),
        include_groups=False
    )
    .dropna()
)

# 统计下降 / 稳定 / 上升
retained_view_trend = pd.Series({
    "decreasing": (retained_view_slopes < -1e-6).sum(),
    "stable": (retained_view_slopes.abs() <= 1e-6).sum(),
    "increasing": (retained_view_slopes > 1e-6).sum()
})

retained_view_percent = (
    retained_view_trend
    / len(retained_view_slopes)
    * 100
).round(2)

print("Valid retained users:", len(retained_view_slopes))

print("\nUser count:")
print(retained_view_trend)

print("\nPercentage:")
print(retained_view_percent)

In [ ]:
retained_cart_purchase_monthly = duckdb.sql("""
SELECT s.user_id, DATE_TRUNC('month', s.event_time) AS month,
 SUM(CASE WHEN s.event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
 SUM(CASE WHEN s.event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases
FROM '../data/sample/session_events.parquet' s
INNER JOIN user_labels u ON s.user_id = u.user_id
WHERE u.churn = 0 AND s.event_time < '2020-03-01'
GROUP BY s.user_id, month ORDER BY s.user_id, month
""").df()
retained_chain_complete = retained_complete.merge(retained_cart_purchase_monthly, on=['user_id', 'month'], how='left', validate='one_to_one')
chain_observed = retained_chain_complete['month'] >= retained_chain_complete['first_seen']
retained_chain_complete.loc[chain_observed, ['carts', 'purchases']] = retained_chain_complete.loc[chain_observed, ['carts', 'purchases']].fillna(0)
assert retained_chain_complete.loc[~chain_observed, ['carts', 'purchases']].isna().all().all()
retained_chain_slopes = retained_chain_complete.groupby('user_id').apply(lambda g: pd.Series({'cart_slope': calculate_slope(g, 'carts'), 'purchase_slope': calculate_slope(g, 'purchases')}), include_groups=False)
retained_chain_slopes['view_slope'] = retained_view_slopes
chain_comparison_rows = []
for group_name, slopes in [('churn', user_slopes.set_index('user_id')), ('retained', retained_chain_slopes)]:
    for metric in ['view_slope', 'cart_slope', 'purchase_slope']:
        valid = slopes[metric].dropna()
        assert len(valid) == (6442 if group_name == 'churn' else 1426)
        counts = {'decreasing': int((valid < -1e-6).sum()), 'stable': int((valid.abs() <= 1e-6).sum()), 'increasing': int((valid > 1e-6).sum())}
        for trend, count in counts.items():
            chain_comparison_rows.append(dict(group=group_name, metric=metric, valid_users=len(valid), trend=trend, count=count, percent=round(count / len(valid) * 100, 2)))
chain_trend_comparison = pd.DataFrame(chain_comparison_rows)
print(chain_trend_comparison.to_string(index=False))
print('All three slopes decreasing (same users):')
for group_name, slopes in [('churn', user_slopes.set_index('user_id')), ('retained', retained_chain_slopes)]:
    valid = slopes[['view_slope', 'cart_slope', 'purchase_slope']].dropna()
    count = int((valid < -1e-6).all(axis=1).sum())
    print(group_name, count, '/', len(valid), '=', round(count / len(valid) * 100, 2), '%')


In [ ]:
value_distribution = user_features[
    ["total_spend", "purchase_count"]
].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]
)

value_distribution

In [ ]:
purchase_freq_distribution = (
    user_features["purchase_count"]
    .value_counts()
    .sort_index()
    .to_frame("user_count")
)

purchase_freq_distribution["percentage"] = (
    purchase_freq_distribution["user_count"]
    / len(user_features)
    * 100
).round(2)

purchase_freq_distribution.head(15)

In [ ]:
user_features["purchase_sessions"].value_counts().sort_index().head(15)

In [ ]:
user_features["purchase_session_rate"] = (
    user_features["purchase_sessions"]
    / user_features["total_sessions"]
)

user_features.shape

In [ ]:
user_features["F_score"] = pd.cut(
    user_features["purchase_sessions"],
    bins=[0, 1, 2, 3, 4, float("inf")],
    labels=[1, 2, 3, 4, 5]
).astype(int)

user_features["F_score"].value_counts().sort_index()

In [ ]:
user_features["R_score"] = pd.qcut(
    user_features["recency_days"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

user_features["R_score"].value_counts().sort_index()

In [ ]:
user_features["M_score"] = pd.qcut(
    user_features["total_spend"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

user_features["M_score"].value_counts().sort_index()

In [ ]:
user_features[
    ["user_id", "R_score", "F_score", "M_score"]
].head(10)

In [ ]:
fm_table = pd.crosstab(
    user_features["F_score"],
    user_features["M_score"]
)

fm_table

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))

sns.heatmap(
    fm_table,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Customer Distribution by F and M Scores")
plt.xlabel("Monetary Score (M)")
plt.ylabel("Frequency Score (F)")

plt.show()

In [ ]:
%pip install matplotlib seaborn

In [ ]:
fm_pct = fm_table.div(
    fm_table.sum(axis=1),
    axis=0
) * 100

plt.figure(figsize=(8, 6))

sns.heatmap(
    fm_pct,
    annot=True,
    fmt=".1f",
    cmap="Blues"
)

plt.title("Monetary Score Distribution within Each F Group")
plt.xlabel("Monetary Score (M)")
plt.ylabel("Frequency Score (F)")

plt.show()

In [ ]:
import numpy as np

conditions = [
    (user_features["M_score"] >= 4) & (user_features["F_score"] >= 3),
    (user_features["M_score"] >= 4) & (user_features["F_score"] <= 2),
    (user_features["M_score"] <= 3) & (user_features["F_score"] >= 3),
    (user_features["M_score"] <= 3) & (user_features["F_score"] <= 2)
]

labels = [
    "High F - High M",
    "Low F - High M",
    "High F - Low M",
    "Low F - Low M"
]

user_features["customer_segment"] = np.select(
    conditions,
    labels,
    default="Unknown"
)

user_features["customer_segment"].value_counts()

In [ ]:
segment_spend = (
    user_features
    .groupby("customer_segment")
    .agg(
        user_count=("user_id", "count"),
        total_spend=("total_spend", "sum"),
        avg_spend=("total_spend", "mean")
    )
)

segment_spend["spend_share"] = (
    segment_spend["total_spend"]
    / segment_spend["total_spend"].sum()
    * 100
).round(2)

segment_spend.round(2)

In [ ]:
user_features.columns.tolist()

In [ ]:
# 旧的重复标签清理已由 one_to_one 合并替代。
assert not any(c.endswith(("_x", "_y")) for c in user_features.columns)


In [ ]:
# 旧的重复标签清理已由 one_to_one 合并替代。
assert not any(c.endswith(("_x", "_y")) for c in user_features.columns)


In [ ]:
segment_churn = (
    user_features
    .groupby("customer_segment")
    .agg(
        user_count=("user_id", "count"),
        churn_count=("churn", "sum"),
        churn_rate=("churn", "mean")
    )
)

segment_churn["churn_rate"] = (
    segment_churn["churn_rate"] * 100
).round(2)

segment_churn

In [ ]:
r_churn = (
    user_features
    .groupby("R_score")
    .agg(
        user_count=("user_id", "count"),
        churn_rate=("churn", "mean")
    )
)

r_churn["churn_rate"] = (
    r_churn["churn_rate"] * 100
).round(2)

r_churn

In [ ]:
segment_r_churn = (
    user_features
    .groupby(["customer_segment", "R_score"])
    .agg(
        user_count=("user_id", "count"),
        churn_rate=("churn", "mean")
    )
    .reset_index()
)

segment_r_churn["churn_rate"] = (
    segment_r_churn["churn_rate"] * 100
).round(2)

segment_r_churn

## RFM Customer Segmentation and Churn Analysis — Key Findings

### 1. Customer Value Distribution

- High-frequency, high-spending customers account for 18.14% of users but contribute 63.95% of historical spending.
- Low-frequency, high-spending customers account for 21.86% of users and contribute 24.71% of historical spending.
- Customer spending is highly concentrated among high-spending segments.

### 2. Purchase Frequency and Churn

- High-frequency customers have a future purchase churn rate of approximately 65%.
- Low-frequency customers have a churn rate of approximately 85%.
- Historical purchase frequency is strongly associated with future purchasing behavior.

### 3. Recency and Churn

- Churn rates decrease as recency scores increase.
- Customers with R=1 have a churn rate of 91.46%, compared with 62.35% for customers with R=5.
- This relationship is observed across all four customer segments.

### 4. Business Implications

- High-frequency, high-spending customers represent an important segment for customer retention.
- Low-frequency, high-spending customers deserve further investigation because of their substantial historical spending and high purchase churn rate.
- Recency and frequency may be useful predictors in subsequent churn prediction models.
- Monetary value can help prioritize customer segments based on historical spending contribution.

### 5. Limitations

- Churn is defined as no purchase during March–April 2020, rather than permanent customer loss.
- The findings describe associations, not causal relationships.
- Customer segments are based on exploratory scoring thresholds and require further validation.
- Retention interventions should be evaluated before assuming they can reduce churn.